# NORTON ResNet-56 / CIFAR-10 (Kaggle)

**TRUOC KHI CHAY:**
1. Settings (ben phai) -> **Accelerator = GPU** (T4 x2 hoac P100). KHONG bat GPU -> train CPU -> qua gio -> exit 137.
2. Add-ons -> **Secrets** -> them `WANDB_API_KEY` (bat cong tac Attach). Neu khong co: chay cell wandb se bao, va them `--no-wandb` o cell run.
3. Datasets da attach:
   - code: `/kaggle/input/datasets/bophaninhthi/norton/OnestageDetectionPunner`
   - ckpt: `/kaggle/input/datasets/bophaninhthi/ckpt-norton/resnet_56 (1).pt`

In [ ]:
# 1) Kiem GPU + cai deps
import torch, subprocess, sys
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CHUA BAT GPU! Settings -> Accelerator = GPU roi chay lai.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorly', 'thop', 'wandb'], check=True)
print('deps OK')

In [ ]:
# 2) WANDB_API_KEY tu Kaggle Secrets (auto-bat wandb). Khong co -> dung --no-wandb o cell run.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('WANDB_API_KEY: loaded from Secrets -> wandb se tu bat')
except Exception as e:
    print('KHONG co WANDB_API_KEY:', e, '\n-> them --no-wandb o cell run neu muon chay khong wandb')

In [ ]:
# 3) Copy code sang /kaggle/working (input read-only) + chdir
import shutil, os
SRC = '/kaggle/input/datasets/bophaninhthi/norton/OnestageDetectionPunner'
CODE = '/kaggle/working/OnestageDetectionPunner'
if not os.path.exists(CODE):
    shutil.copytree(SRC, CODE)
os.chdir(CODE)
print('cwd:', os.getcwd())

In [ ]:
# 4) (Tuy chon) SMOKE TEST nhanh 2+2 epoch de chac pipeline chay, log hien ngay.
#    Chay cell nay truoc; neu OK thi bo qua, sang cell 5 chay that.
!cd /kaggle/working/OnestageDetectionPunner && PYTHONUNBUFFERED=1 python -u norton_cifar10.py \
    --dense-checkpoint "/kaggle/input/datasets/bophaninhthi/ckpt-norton/resnet_56 (1).pt" \
    --data-path ./cifar10-data \
    --decompose-finetune-epochs 2 --prune-finetune-epochs 2 \
    --rank 6 --compress-rate "[0.]+[0.18]*29" \
    --batch-size 256 --workers 2 --no-wandb \
    --output-dir /kaggle/working/out_smoke

In [ ]:
# 5) CHAY THAT (wandb auto-bat neu co WANDB_API_KEY). PYTHONUNBUFFERED + -u => log stream.
#    epochs vua phai cho khop gio Kaggle; tang neu con thoi gian.
!cd /kaggle/working/OnestageDetectionPunner && PYTHONUNBUFFERED=1 python -u norton_cifar10.py \
    --dense-checkpoint "/kaggle/input/datasets/bophaninhthi/ckpt-norton/resnet_56 (1).pt" \
    --data-path ./cifar10-data \
    --decompose-finetune-epochs 200 --prune-finetune-epochs 200 \
    --rank 6 --compress-rate "[0.]+[0.18]*29" \
    --criterion pabs --batch-size 256 --workers 2 \
    --wandb-project resnet56-cifar10-norton \
    --wandb-run-name resnet56-norton-r6-cpr018 \
    --output-dir /kaggle/working/out_norton